# toonpy demo

This notebook demonstrates JSON ↔ TOON conversion and token estimates.


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from toontools import from_toon, to_toon, suggest_tabular


In [2]:
example_json = Path("../examples/example1.json").read_text()
data = json.loads(example_json)
data


{'crew': [{'id': 1, 'name': 'Luz', 'role': 'Light glyph'},
  {'id': 2, 'name': 'Amity', 'role': 'Abomination strategist'}],
 'active': True,
 'ship': {'name': 'Owl House', 'location': 'Bonesborough'}}

In [3]:
toon_text = to_toon(data)
toon_text


'crew[2]{id,name,role}:\n  1,Luz,"Light glyph"\n  2,Amity,"Abomination strategist"\nactive: true\nship:\n  name: "Owl House"\n  location: Bonesborough\n'

In [4]:
round_trip = from_toon(toon_text)
assert round_trip == data
round_trip


{'crew': [{'id': 1, 'name': 'Luz', 'role': 'Light glyph'},
  {'id': 2, 'name': 'Amity', 'role': 'Abomination strategist'}],
 'active': True,
 'ship': {'name': 'Owl House', 'location': 'Bonesborough'}}

In [6]:
# Token comparison: try tiktoken first, fallback to character count
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    json_tokens = len(enc.encode(example_json))
    toon_tokens = len(enc.encode(toon_text))
    result = {
        "method": "tiktoken (cl100k_base)",
        "json_tokens": json_tokens,
        "toon_tokens": toon_tokens,
        "savings": json_tokens - toon_tokens,
        "savings_percent": round((json_tokens - toon_tokens) / json_tokens * 100, 1) if json_tokens > 0 else 0,
    }
except ImportError:
    # Fallback to character count if tiktoken not available
    json_chars = len(example_json)
    toon_chars = len(toon_text)
    result = {
        "method": "character_count (tiktoken not installed)",
        "json_chars": json_chars,
        "toon_chars": toon_chars,
        "savings": json_chars - toon_chars,
        "savings_percent": round((json_chars - toon_chars) / json_chars * 100, 1) if json_chars > 0 else 0,
        "note": "Install tiktoken for accurate token counts: pip install tiktoken",
    }
except Exception as exc:
    result = {
        "method": "error",
        "error": str(exc),
        "fallback": "character_count",
        "json_chars": len(example_json),
        "toon_chars": len(toon_text),
    }

result


{'method': 'tiktoken (cl100k_base)',
 'json_tokens': 83,
 'toon_tokens': 50,
 'savings': 33,
 'savings_percent': 39.8}